# Frozen E1 WiSE-FT + DINOv2 crop fusion
No gradient training. Preserve original CLIP prompts, preprocessing, candidate order and splits. Defaults to audit-only; full GPU results remain pending.

## Setup

In [ ]:
from pathlib import Path
import sys, json, torch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common/dataset.py").is_file())
sys.path.insert(0, str(ROOT))
from approach3_dino_fusion.run import execute, check_config
from approach3_ood_backbones.data import shared_frames
CONFIG = json.loads((ROOT / "approach3_dino_fusion/config.json").read_text())
check_config(CONFIG)
E1_PATH = ROOT / "checkpoints/dino_fusion_inputs/E1_wise.pt"
RUN = False
STAGE = "smoke"  # then verify, then full on RTX 5090
ALLOW_DOWNLOAD = False
print(torch.__version__, "CUDA:", torch.cuda.is_available())

## Data
Prototypes use the original 7,223 optimization-training images only. Original validation and both official tests remain unchanged.

In [ ]:
frames = shared_frames()
print({name: len(frame) for name, frame in frames.items()})
assert not set(frames["train"].image) & set(frames["validation"].image)

## Model
Load E1 using existing team checkpoint functions. Require original stored prompts and 333 candidate rows. Full runs verify recorded E1 metrics and robustness before loading frozen facebook/dinov2-base.

In [ ]:
print("E1 checkpoint:", E1_PATH)
print("DINO:", CONFIG["dino_repository"])
print("Weights are loaded only when RUN=True.")

## Training
No optimizer, backward pass or neural updates. Build each crop prototype as the normalized mean of unit DINO CLS embeddings. Freeze the fusion coefficient before any test scoring.

In [ ]:
print("Neural training images: 0; prototype source images:", len(frames["train"]))
print("Fixed fusion:", CONFIG["clip_fraction"], "CLIP +", CONFIG["dino_fraction"], "DINO")

## Evaluation
First smoke, then verify, then full. Smoke is not E1 reproduction and uses one image per crop plus two test images per split. Full mode rechecks E1, stops on mismatch, and only then extracts all prototype features and runs fusion. DINO-only accuracy is crop accuracy, never disease/pair accuracy.

In [ ]:
run_path = None
if RUN:
    run_path = execute(CONFIG, E1_PATH, stage=STAGE, allow_download=ALLOW_DOWNLOAD)
else:
    print("Audit only. No model weights downloaded or full evaluation performed.")

## Save
Each execution saves a new unique experiment directory: protocol, reproduction checks, model metadata, crop prototypes/embeddings, per-image predictions/scores, comparison tables, OOD breakdown and confusion matrices.

In [ ]:
print("Results:", run_path if run_path else "No model run in audit-only mode")